# Intent Resolution Evaluator Sample
**IMPORTANT NOTE**<br/>
- These samples use `GPT-4.1 mini` because `azure-ai-evaluation 1.18.3` local agent evaluators send the legacy max_tokens parameter.
- Newer GPT-5 deployments require `max_completion_tokens` and aren't compatible with this local evaluator path.
- For managed evaluations with newer judge models, see 4 - cloud evaluation.

## Variables, Constants and Libraries definition

In [1]:
import os, sys
from openai import AzureOpenAI
from dotenv import load_dotenv # requires python-dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

if not load_dotenv("./../credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

openai_api_version    = os.environ["AZURE_OPENAI_API_VERSION"]
azure_openai_endpoint =  os.environ["AZURE_OPENAI_ENDPOINT"]

# gpt-4.1-mini is the latest working model because later models require max_completion_tokens, while this evaluator sends max_tokens
azure_evaluation_compatible_deployment_name= os.environ["AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

print(f"azure_openai_endpoint: {azure_openai_endpoint}")
print(f"azure_evaluation_compatible_deployment_name: {azure_evaluation_compatible_deployment_name}")
print(f"openai_api_version: {openai_api_version}")

azure_openai_endpoint: https://mm-ai-upskilling-project-resourc.openai.azure.com/
azure_evaluation_compatible_deployment_name: gpt-4.1-mini
openai_api_version: 2025-04-01-preview


## Initialize Intent Resolution Evaluator

The Intent Resolution evaluator measures how well an agent has identified and resolved the user intent.
The scoring is on a 1-5 integer scale and is as follows:

  - Score 1: Response completely unrelated to user intent
  - Score 2: Response minimally relates to user intent
  - Score 3: Response partially addresses the user intent but lacks complete details
  - Score 4: Response addresses the user intent with moderate accuracy but has minor inaccuracies or omissions
  - Score 5: Response directly addresses the user intent and fully resolves it

The evaluation requires the following inputs:

  - Query    : The user query. Either a string with a user request or a list of messages with previous requests from the user and responses from the assistant, potentially including a system message.
  - Response : The response to be evaluated. Either a string or a message with the response from the agent to the last user query.

There is a third optional parameter:
  - ToolDefinitions : The list of tool definitions the agent can call. This may be useful for the evaluator to better assess if the right tool was called to resolve a given intent.

In [2]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.ai.evaluation import IntentResolutionEvaluator
from pprint import pprint
from dotenv import load_dotenv # requires python-dotenv
import warnings

warnings.filterwarnings("ignore")

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment=azure_evaluation_compatible_deployment_name,
    api_version=openai_api_version,
)

intent_resolution_evaluator = IntentResolutionEvaluator(
    model_config,
    credential=credential,
)

# Print some constants
print(f'openai endpoint: <{model_config["azure_endpoint"]}>')
print(f'azure deployment name: <{model_config["azure_deployment"]}>')
print(f'openai api version: <{model_config["api_version"]}>')

Class IntentResolutionEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


openai endpoint: <https://mm-ai-upskilling-project-resourc.openai.azure.com/>
azure deployment name: <gpt-4.1-mini>
openai api version: <2025-04-01-preview>


### Samples

#### Evaluating query and response as string

In [3]:
#Success example. Intent is identified and understood and the response correctly resolves user intent
result = intent_resolution_evaluator(
    query="What are the opening hours of the Eiffel Tower?",
    response="Opening hours of the Eiffel Tower are 9:00 AM to 11:00 PM.", # then, try using 11:00 AM instead of 11:00 PM
)

pprint(result)

Conversation history could not be parsed; falling back to raw input. Evaluator accuracy will degrade. Input shape: type=str len=47. Error: 'str' object has no attribute 'get'
Empty agent response extracted, likely due to input schema change. Falling back to original response. type=str len=58


{'intent_resolution': 5.0,
 'intent_resolution_passed': True,
 'intent_resolution_properties': {'completion_tokens': 59,
                                  'finish_reason': 'stop',
                                  'model': 'gpt-4.1-mini-2025-04-14',
                                  'prompt_tokens': 2061,
                                  'sample_input': '[{"role": "user", '
                                                  '"content": "{\\"query\\": '
                                                  '\\"What are the opening '
                                                  'hours of the Eiffel '
                                                  'Tower?\\", \\"response\\": '
                                                  '\\"Opening hours of the '
                                                  'Eiffel Tower are 9:00 AM to '
                                                  '11:00 PM.\\", '
                                                  '\\"tool_definitions\\": '
           

In [4]:
#Failure example. Even though intent is correctly identified, the response does not resolve the user intent
result = intent_resolution_evaluator(
    query="What is the opening hours of the Eiffel Tower?",
    response="Please check the official website for the up-to-date information on Eiffel Tower opening hours.",
)

pprint(result)

Conversation history could not be parsed; falling back to raw input. Evaluator accuracy will degrade. Input shape: type=str len=46. Error: 'str' object has no attribute 'get'
Empty agent response extracted, likely due to input schema change. Falling back to original response. type=str len=95


{'intent_resolution': 3.0,
 'intent_resolution_passed': True,
 'intent_resolution_properties': {'completion_tokens': 73,
                                  'finish_reason': 'stop',
                                  'model': 'gpt-4.1-mini-2025-04-14',
                                  'prompt_tokens': 2059,
                                  'sample_input': '[{"role": "user", '
                                                  '"content": "{\\"query\\": '
                                                  '\\"What is the opening '
                                                  'hours of the Eiffel '
                                                  'Tower?\\", \\"response\\": '
                                                  '\\"Please check the '
                                                  'official website for the '
                                                  'up-to-date information on '
                                                  'Eiffel Tower opening '
          

#### Evaluating query and response as list of messages

In [5]:
query = [
    {
        "role": "system",
        "content": "You are a friendly and helpful customer service agent."
    },
    {
        "createdAt": "2025-03-14T06:14:20Z",
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Hi, I need help with the last 2 orders on my account #888. Could you please update me on their status?"
            }
        ]
    }
]

response = [
    {
        "createdAt": "2025-03-14T06:14:30Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Hello! Let me quickly look up your account details."
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:35Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_001",
                "name": "get_orders",
                "arguments": {
                    "account_number": "888"
                }
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:40Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_001",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": "[{ \"order_id\": \"123\" }, { \"order_id\": \"124\" }]"
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:45Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Thanks for your patience. I see two orders on your account. Let me fetch the details for both."
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:50Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_002",
                "name": "get_order",
                "arguments": {
                    "order_id": "123"
                }
            },
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_003",
                "name": "get_order",
                "arguments": {
                    "order_id": "124"
                }
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:55Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_002",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": "{ \"order\": { \"id\": \"123\", \"status\": \"shipped\", \"delivery_date\": \"2025-03-15\" } }"
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:15:00Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_003",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": "{ \"order\": { \"id\": \"124\", \"status\": \"delayed\", \"expected_delivery\": \"2025-03-20\" } }"
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:15:05Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "The order with ID 123 has been shipped and is expected to be delivered on March 15, 2025. However, the order with ID 124 is delayed and should now arrive by March 20, 2025. Is there anything else I can help you with?"
            }
        ]
    }
]

#please note that the tool definitions are not strictly required, and that some of the tools below are not used in the example above and that is ok.
#if context length is a concern you can remove the unused tool definitions or even the tool definitions altogether as the impact to the intent resolution evaluation is usual minimal.
tool_definitions = [
    {
        "name": "get_orders",
        "description": "Get the list of orders for a given account number.",
        "parameters": {
            "type": "object",
            "properties": {
                "account_number": {
                    "type": "string",
                    "description": "The account number to get the orders for."
                }
            }
        }
    },
    {
        "name": "get_order",
        "description": "Get the details of a specific order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID to get the details for."
                }
            }
        }
    },
    {
        "name": "initiate_return",
        "description": "Initiate the return process for an order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID for the return process."
                }
            }
        }
    },
    {
        "name": "update_shipping_address",
        "description": "Update the shipping address for a given account.",
        "parameters": {
            "type": "object",
            "properties": {
                "account_number": {
                    "type": "string",
                    "description": "The account number to update."
                },
                "new_address": {
                    "type": "string",
                    "description": "The new shipping address."
                }
            }
        }
    }
]

result = intent_resolution_evaluator(query            = query,
                                     response         = response,
                                     tool_definitions = tool_definitions,
                                    )
pprint(result)

{'intent_resolution': 5.0,
 'intent_resolution_passed': True,
 'intent_resolution_properties': {'completion_tokens': 63,
                                  'finish_reason': 'stop',
                                  'model': 'gpt-4.1-mini-2025-04-14',
                                  'prompt_tokens': 2153,
                                  'sample_input': '[{"role": "user", '
                                                  '"content": "{\\"query\\": '
                                                  '\\"User turn 1:\\\\n  Hi, I '
                                                  'need help with the last 2 '
                                                  'orders on my account #888. '
                                                  'Could you please update me '
                                                  'on their '
                                                  'status?\\\\n\\\\n\\", '
                                                  '\\"response\\": \\"Hello! '
       

### Evaluating an agent conversation loaded from disk

In [6]:
import json

def load_conversations(filename):
    with open(filename, 'r') as file:
        lines = file.readlines()
        parsed_conversations = [json.loads(line) for line in lines]
    print(f"Loaded {len(parsed_conversations)} conversations from {filename}.\n")
    return parsed_conversations

conversations_filename = r'sample_synthetic_conversations.jsonl'

#this loads 90 conversations from the file sample_synthetic_conversations.jsonl
sample_conversations = load_conversations(conversations_filename)

#get the first conversation from the loaded conversations
conversation = sample_conversations[10]

pprint(conversation)

Loaded 90 conversations from sample_synthetic_conversations.jsonl.

{'messages': [{'content': 'You are a healthcare support agent assisting '
                          'patients with appointment scheduling, prescription '
                          'refills, test results, and general health '
                          'inquiries.',
               'createdAt': 1741618727,
               'role': 'system'},
              {'content': [{'text': 'Can you update my health records? I '
                                    'recently had a lab test and need the '
                                    'results added to my profile.',
                            'type': 'text'}],
               'createdAt': 1741618732,
               'role': 'user'},
              {'content': [{'text': 'I’ll update your health records with the '
                                    'new lab test results. One moment, please.',
                            'type': 'text'}],
               'createdAt': 1741618737,
         

### Prepare a local conversation for evaluation



The records loaded from the JSONL file already contain the agent conversation under `messages` and the available tool definitions under `tools`. The `IntentResolutionEvaluator` expects these values as three separate inputs:



- `query`: the conversation history up to and including the most recent user message;

- `response`: the assistant and tool messages generated after that user message;

- `tool_definitions`: the tools available to the agent.



The next cell locates the most recent user turn, splits the conversation at that point, and evaluates the resulting response.



> **About `AIAgentConverter`:** it is not needed for an already-loaded local dictionary. In `azure-ai-evaluation 1.18.3`, `AIAgentConverter` uses an `AIProjectClient` to retrieve and convert Microsoft Foundry agent data identified by a `thread_id` and `run_id`. That cloud-backed workflow is covered separately in the cloud evaluation section.

In [7]:
messages = conversation["messages"]

# Evaluate the agent response to the most recent user turn.
last_user_index = max(
    index for index, message in enumerate(messages)
    if message["role"] == "user"
)

query = messages[:last_user_index + 1]
response = messages[last_user_index + 1:]
tool_definitions = conversation.get("tools")

result = intent_resolution_evaluator(
    query=query,
    response=response,
    tool_definitions=tool_definitions,
)

pprint(result)

{'intent_resolution': 5.0,
 'intent_resolution_passed': True,
 'intent_resolution_properties': {'completion_tokens': 56,
                                  'finish_reason': 'stop',
                                  'model': 'gpt-4.1-mini-2025-04-14',
                                  'prompt_tokens': 2137,
                                  'sample_input': '[{"role": "user", '
                                                  '"content": "{\\"query\\": '
                                                  '\\"User turn 1:\\\\n  Can '
                                                  'you update my health '
                                                  'records? I recently had a '
                                                  'lab test and need the '
                                                  'results added to my '
                                                  'profile.\\\\n\\\\nAgent '
                                                  'turn 1:\\\\n  I\\\\u2019ll '
      